In [ ]:
import sys

if "google.colab" in sys.modules:
    import os

    repo = "/content/lmbike_workshop_gimbal"
    sim = "/content/lmbike_workshop_gimbal/bike_mujoco_sim"
    if not os.path.isdir(repo):
        !git clone https://github.com/innowing-robotics-interns/lmbike_workshop_gimbal.git /content/lmbike_workshop_gimbal
    %cd /content/lmbike_workshop_gimbal/bike_mujoco_sim
    if sim not in sys.path:
        sys.path.insert(0, sim)

    # Colab's Jupyter must stay the copy it shipped with. requirements.txt
    # does not list Jupyter. The constraint pins whatever Jupyter packages
    # are already installed so pip cannot replace them.
    import importlib.metadata as metadata
    protected = {
        "notebook", "jupyter", "jupyterlab", "jupyter-client", "jupyter-core",
        "jupyter-server", "ipykernel", "ipython", "nbformat", "nbclient",
        "nbconvert", "nbclassic",
    }
    pinned = []
    for dist in metadata.distributions():
        name = dist.metadata["Name"]
        if name and name.lower().replace("_", "-") in protected:
            pinned.append(f"{name}=={dist.version}")
    with open("/tmp/colab-jupyter-constraints.txt", "w", encoding="utf-8") as fh:
        fh.write("\n".join(pinned) + "\n")
    !pip install -q --upgrade-strategy only-if-needed -r requirements.txt -c /tmp/colab-jupyter-constraints.txt


# Lesson 3. Tuning the balancing loop, one gain at a time

At the end of Lesson 2, the bicycle still fell with the gains in `configs/tutorial.yaml`, even though the balancing loop was working hard. In this lesson you will fix that yourself. You will change the balancing-loop gains, measure what each change does, and end up with your own set of gains that keeps the bicycle upright and calm. Then you will save your gains into `tutorial.yaml`, so that they are kept for the later lessons.

Choosing gains like this is called **tuning**. The method is simple, and engineers use it everywhere:

1. Change one number.
2. Run the bicycle.
3. Measure the result.
4. Keep the change only if it helped.

Changing only one number at a time is important. If you changed two numbers at once and the ride improved, you would not know which change caused the improvement.

This is a longer lesson, because each part of PID gets its own experiment. Here is the route:

| Step | What you do | What you learn |
| --- | --- | --- |
| 1–2 | Prepare the notebook and its measuring tools. | How a ride is turned into a few numbers you can compare. |
| 3 | Ride the starting gains from Lesson 2 again. | Where we are starting from. |
| 4 | Use P on its own. | Why P alone cannot balance this bicycle. |
| 5 | Add D. | How D catches the fall, and roughly how much D is needed. |
| 6 | Use far too much D. | Why more is not always better. |
| 7 | Look closely at a ride that holds. | The small lean that P and D leave behind. |
| 8 | Add I. | How I removes that leftover lean, and what too much I does. |
| 9 | Tune all of it yourself. | Putting the method together. |
| 10 | Save your gains. | Keeping your work. |
| 11 | Compare with a reference set. | How your result compares with a set found by a much longer search. |

Throughout the lesson, the bicycle, the motors and the program stay exactly the same. Only the balancing-loop gains change.

## Learning objectives

By the end of this lesson you will be able to:

1. Tell from a lean graph and a short report whether the bicycle fell, held, or held with problems.
2. Show by experiment that the P part on its own cannot balance this bicycle.
3. Find roughly how large the D gain has to be before the bicycle stays up, and recognize when D is too large.
4. Explain what the leftover lean is, and use the I gain to remove it without overdoing it.
5. Tune a full set of gains yourself and save it in `configs/tutorial.yaml`.

## Glossary

This lesson builds on P, I and D from Lesson 2. As a reminder: P steers against how far the bicycle is leaning right now, I steers against a lean that has lasted, and D steers against how fast the lean is changing. In this lesson we often shorten the gain names, the way engineers do:

| Short name | Setting in the file | Part |
| --- | --- | --- |
| Kp | `pid_kp_roll` | P gain |
| Ki | `pid_ki_roll` | I gain |
| Kd | `pid_kd_roll` | D gain |
| Kp_steer | `pid_kp_steer` | the steer term from Lesson 2 |

A few more words describe what you will see:

| Term | Meaning |
| --- | --- |
| Tuning | Choosing a controller's gains by changing one at a time and checking the result after each change. |
| Fallen | The bicycle's lean passed 45°, the fall mark from Lessons 1 and 2. At that point it is too far over to recover. |
| Holding | The bicycle stayed up for the whole ride. |
| Leftover lean | A small, steady lean that remains after the bicycle has settled down, instead of the lean returning exactly to 0°. Engineers call this the **steady-state error**. |
| Overshoot | Going past the target and out the other side, like a swing that passes the bottom and keeps going. |
| Steering shake | The handlebars flicking quickly back and forth even though the bicycle is not falling. It is a sign that the loop is over-reacting. |
| Trade-off | A choice where improving one thing makes another thing worse, so you have to look for a good balance between them. |
| Outer gains | Kp, Ki, Kd and Kp_steer, the balancing-loop gains. These are the numbers you change in this lesson. |
| Inner gains | `steer_kp` and `steer_ki`, the gains of the small loop inside the steering motor. Keep them as they are. |

## Step 1. Start the lesson

As in the earlier lessons, the first code cell prepares the notebook. Run it now, and continue when you see the line that begins with `project_root:`.

In [ ]:
import os
import sys

def choose_mujoco_gl(platform, display, existing):
    if existing:
        return existing
    if str(platform).startswith("linux") and not display:
        return "egl"
    return None

chosen = choose_mujoco_gl(sys.platform, os.environ.get("DISPLAY"), os.environ.get("MUJOCO_GL"))
if chosen and "MUJOCO_GL" not in os.environ:
    os.environ["MUJOCO_GL"] = chosen

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import replace

from sim.config import load_config
from sim.loop import make_sim, run_headless, summarize
from sim.pid_controller import RollSteerPID, apply_speed_refs

np.set_printoptions(precision=4, suppress=True)
print("project_root:", project_root)
print("platform:", sys.platform, "MUJOCO_GL:", os.environ.get("MUJOCO_GL", "(unset)"))

## Step 2. Create the measuring tools

In this lesson you will run many rides and compare them. Watching each one in the window is useful, but your eyes cannot tell a 0.3° lean from a 0° lean, and they cannot compare ten rides at once. So engineers turn each ride into a few numbers, and compare the numbers.

The next cell defines three shortcuts, called **functions**. A function is a named recipe: you write the steps once, give them a name, and from then on you can use the name instead of the steps.

- `ride` runs one ride without a window and prints a short report. The first line of the report always shows all four gains that were used, so you never have to guess which numbers produced which result.
- `plot_roll` draws a graph of lean against time, like the graphs in Lessons 1 and 2. It can also draw a **zoomed** graph, which shows only the part of the ride after the start-up, with the up-and-down scale stretched so that small leans become visible.
- `plot_steer` draws the steering command against time, like the lower graph in Lesson 2.

The report from `ride` gives one of two results.

If the bicycle **fell**, the report says when the fall happened and how far the bicycle leaned.

If the bicycle **held**, the report gives three measurements:

| Measurement | What it measures | What a good value looks like |
| --- | --- | --- |
| largest lean | The biggest lean during the whole ride, in degrees. It is usually reached in the first second, while the bicycle is speeding up. | Small. Under 10° is good. |
| leftover lean | The average lean over the last 3 seconds of the ride, after the bicycle has settled. The sign tells you which side it settled on. | Close to 0. Between −0.10° and +0.10° is good. |
| steering shake | How much the steering command jumps around after the start-up, in rad/s. A calm loop has a tiny value. A loop that keeps flicking the handlebars back and forth has a large one. | Tiny. Under 0.10 rad/s is good. |

Every ride in this lesson lasts 8 seconds, so that there is time for the bicycle to settle before the leftover lean is measured.

Running this cell only creates the shortcuts. Nothing rides yet, and nothing is printed. Simply run it once.

In [ ]:
tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
RIDE_SECONDS = 8.0


def gains_text(cfg):
    return (
        f"Kp={cfg.pid_kp_roll:.4f}  Ki={cfg.pid_ki_roll:.4f}  "
        f"Kd={cfg.pid_kd_roll:.4f}  Kp_steer={cfg.pid_kp_steer:.4f}"
    )


def with_gains(kp, ki, kd, kp_steer=1.1164):
    return replace(
        load_config(tutorial_yaml),
        pid_kp_roll=float(kp),
        pid_ki_roll=float(ki),
        pid_kd_roll=float(kd),
        pid_kp_steer=float(kp_steer),
    )


def ride(cfg, seconds=RIDE_SECONDS):
    cfg = replace(cfg, duration=float(seconds))
    sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
    history = run_headless(sim, duration=seconds)
    time_s = np.asarray(history["time"], dtype=float)
    roll_deg = np.degrees(np.asarray(history["roll"], dtype=float))
    steer_cmd = np.asarray(history["steer_rate_ref"], dtype=float)
    tip = np.where(np.abs(roll_deg) > 45.0)[0]
    report = {
        "largest_lean": float(np.max(np.abs(roll_deg))),
        "fall_time": float(time_s[tip[0]]) if len(tip) else None,
        "leftover_lean": float(np.mean(roll_deg[time_s > seconds - 3.0])),
        "steering_shake": float(np.std(steer_cmd[time_s > 2.0])),
    }
    print(gains_text(cfg))
    if report["fall_time"] is not None:
        print(f"    fell at {report['fall_time']:.2f} s, largest lean {report['largest_lean']:.1f} deg")
    else:
        print(
            f"    held.  largest lean {report['largest_lean']:.2f} deg,  "
            f"leftover lean {report['leftover_lean']:+.2f} deg,  "
            f"steering shake {report['steering_shake']:.2f} rad/s"
        )
    return history, report


def plot_roll(history, title, ax=None, zoom=False, label=None):
    time_s = np.asarray(history["time"], dtype=float)
    roll_deg = np.degrees(np.asarray(history["roll"], dtype=float))
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(8, 3.2), constrained_layout=True)
    if zoom:
        keep = time_s > 2.0
        ax.plot(time_s[keep], roll_deg[keep], label=label)
        ax.set_ylim(-1.5, 1.5)
        ax.set_ylabel("roll (deg), zoomed")
    else:
        ax.plot(time_s, roll_deg, label=label)
        ax.axhline(45.0, color="0.5", linestyle="--", linewidth=0.8)
        ax.axhline(-45.0, color="0.5", linestyle="--", linewidth=0.8)
        ax.set_ylabel("roll (deg)")
    ax.axhline(0.0, color="0.7", linewidth=0.8)
    ax.set_xlabel("time (s)")
    ax.set_title(title)
    if label is not None:
        ax.legend()
    if own:
        plt.show()


def plot_steer(history, title, ax):
    ax.plot(history["time"], history["steer_rate_ref"])
    ax.set_ylim(-13, 13)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("steering command (rad/s)")
    ax.set_title(title)

## Step 3. Where we are starting from

Before changing anything, it is good practice to measure the starting point, so that you can tell later whether your changes helped. The next cell rides `configs/tutorial.yaml` exactly as it is saved, which is the ride you watched at the end of Lesson 2.

Run the cell and read the report. The first line shows the four starting gains. The second line shows the result. You saw this ride fall in Lesson 2, so the report should agree.

In [ ]:
start_cfg = load_config(tutorial_yaml)
start_history, start_report = ride(start_cfg)

## Step 4. Try P on its own

The simplest possible balancing loop uses only P: steer in proportion to how far the bicycle is leaning. It is natural to think that if P is not strong enough, a bigger Kp will fix things. Let's test that idea.

For this experiment, Ki and Kd are both set to 0, so only P is working, together with the small steer term, which stays at 1.1164 for the whole lesson. Then we try a range of Kp values, from small to large: 2, 4, 6, 7.5 and 9.

There are two cells:

1. The **window cell** plays the ride with the largest value, Kp = 9, so that you can see the strongest P-only loop in action. Watch it, then close the window.
2. The **graph cell** runs all five Kp values without a window. It prints a report for each one, and then draws the lean graph for the last one, Kp = 9. The graph's title only tells you which gains were used.

Read the five reports from top to bottom. Did making Kp bigger and bigger change the outcome? Did it change *when* the fall happened?

If a window cell stops with a message saying MuJoCo was already imported, use **Kernel → Restart Kernel**. Run the window cell again, then run the Step 1 and Step 2 cells once more before you run the graph cell.

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
cfg = replace(
    cfg,
    pid_kp_roll=9.0,
    pid_ki_roll=0.0,
    pid_kd_roll=0.0,
    pid_kp_steer=1.1164,
    duration=8.0,
)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print(
    f"Kp={cfg.pid_kp_roll:.4f}  Ki={cfg.pid_ki_roll:.4f}  "
    f"Kd={cfg.pid_kd_roll:.4f}  Kp_steer={cfg.pid_kp_steer:.4f}"
)
print("P only")
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")

In [ ]:
for kp in (2.0, 4.0, 6.0, 7.5, 9.0):
    history, report = ride(with_gains(kp=kp, ki=0.0, kd=0.0))
plot_roll(history, "Kp = 9, Ki = 0, Kd = 0")

### What Step 4 shows

Every P-only ride fell, and at about the same moment, roughly half a second in. Making Kp four times larger hardly changed anything.

Here is why. P only reacts to how far the bicycle has *already* leaned. At the very start of a fall the lean is still small, so P's correction is small too, even with a large Kp. Meanwhile, the bicycle is speeding up as it tips over. By the time the lean is big enough for P to push hard, the bicycle is falling too fast to be caught. A bigger Kp pushes harder at every angle, but P still has no idea how fast the bicycle is falling, so it is always too late.

What the loop is missing is a part that reacts to *how fast* the bicycle is falling. That is D.

## Step 5. Add D, the brake on a fast fall

D reacts to how fast the lean is changing. At the very start of a fall, the lean is still small but the tipping speed is already high, so D can push back early, before the fall gets away.

In this experiment, **Kp stays fixed at 7.5** and **Ki stays at 0**. Only Kd changes. This is the one-number-at-a-time rule in action: if the outcome changes, we know D caused it. Every report's first line shows the current Kp next to the Kd being tried, so you can always check what P was doing during that ride.

There are three cells:

1. The **first window cell** plays Kp = 7.5 with Kd = 5. Watch it, then close the window.
2. The **second window cell** plays Kp = 7.5 with Kd = 8. Watch it, then close the window.
3. The **graph cell** runs Kd = 3, 5, 6, 8 and 11, all with Kp = 7.5, prints a report for each, and draws the Kd = 5 and Kd = 8 rides side by side.

Read the five reports in order. Look for the point where the result changes from `fell` to `held`. That is roughly the Kd at which the brake becomes strong enough. Then, among the rides that held, look at the largest lean. What happens to it as Kd grows?

If a window cell stops with a message saying MuJoCo was already imported, use **Kernel → Restart Kernel**. Run the window cell again, then run the Step 1 and Step 2 cells once more before you run the graph cell.

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
cfg = replace(
    cfg,
    pid_kp_roll=7.5,
    pid_ki_roll=0.0,
    pid_kd_roll=5.0,
    pid_kp_steer=1.1164,
    duration=8.0,
)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print(
    f"Kp={cfg.pid_kp_roll:.4f}  Ki={cfg.pid_ki_roll:.4f}  "
    f"Kd={cfg.pid_kd_roll:.4f}  Kp_steer={cfg.pid_kp_steer:.4f}"
)
print("Kp fixed at 7.5, trying Kd = 5")
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
cfg = replace(
    cfg,
    pid_kp_roll=7.5,
    pid_ki_roll=0.0,
    pid_kd_roll=8.0,
    pid_kp_steer=1.1164,
    duration=8.0,
)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print(
    f"Kp={cfg.pid_kp_roll:.4f}  Ki={cfg.pid_ki_roll:.4f}  "
    f"Kd={cfg.pid_kd_roll:.4f}  Kp_steer={cfg.pid_kp_steer:.4f}"
)
print("Kp fixed at 7.5, trying Kd = 8")
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")

In [ ]:
kp_now = 7.5
print(f"Kd sweep with Kp fixed at {kp_now:.4f} and Ki = 0")
histories = {}
for kd in (3.0, 5.0, 6.0, 8.0, 11.0):
    histories[kd], report = ride(with_gains(kp=kp_now, ki=0.0, kd=kd))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True, constrained_layout=True)
plot_roll(histories[5.0], f"Kp = {kp_now}, Kd = 5", ax=axes[0])
plot_roll(histories[8.0], f"Kp = {kp_now}, Kd = 8", ax=axes[1])
plt.show()

### Try it: your own Kd

Now pick your own Kd values and see where the boundary between falling and holding lies. The next two cells use the same `kd_try` number: the first shows the ride in a window, and the second measures it and draws it. Kp stays at 7.5 and Ki stays at 0.

1. Set `kd_try = 7.0` in **both** cells. Run the window cell, watch, close the window, then run the graph cell.
2. Repeat with `kd_try = 4.0` in both cells.
3. Repeat with any value you are curious about, for example `15.0`.

Before each run, predict whether the bicycle will hold, and compare the report with your prediction.

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


# Keep kd_try the same as in the graph cell below.
kd_try = 7.0
kp_now = 7.5
tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
cfg = replace(cfg, pid_kp_roll=kp_now, pid_ki_roll=0.0, pid_kd_roll=float(kd_try), pid_kp_steer=1.1164, duration=8.0)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print(
    f"Kp={cfg.pid_kp_roll:.4f}  Ki={cfg.pid_ki_roll:.4f}  "
    f"Kd={cfg.pid_kd_roll:.4f}  Kp_steer={cfg.pid_kp_steer:.4f}"
)
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")

In [ ]:
# Keep kd_try the same as in the window cell above.
kd_try = 7.0
kp_now = 7.5
history, report = ride(with_gains(kp=kp_now, ki=0.0, kd=kd_try))
plot_roll(history, f"Kp = {kp_now}, Ki = 0, Kd = {kd_try}")

### What Step 5 shows

With Kp at 7.5, a Kd of 5 or less still fell, although a little later than with less D. At around Kd = 6 the brake became strong enough to stop the fall, and every larger Kd you tried held. Among the rides that held, a larger Kd gave a smaller largest lean, because the fall at the start was braked harder.

You may remember that in Lesson 2, `tutorial.yaml` had Kd = 6 and fell. Here, Kd = 6 held. The difference is Kp: it was 5 in Lesson 2 and 7.5 here. The parts of a PID controller work as a team, so the right value for one gain depends on the others. A Kd that is just enough when Kp is 7.5 can be too little when Kp is 5. That is why every report in this lesson prints all four gains together: a gain value only means something alongside the other gains it was tested with.

If a larger Kd gives a smaller lean, should we simply make Kd as large as possible? The next step tests that.

## Step 6. Use far too much D

So far, more D has always looked better. Let's push it much further, to Kd = 30, keeping Kp at 7.5 and Ki at 0.

This time, watch the **handlebars** rather than the lean. The lean will look fine. The interesting part is what the steering is doing.

There are two cells:

1. The **window cell** plays Kp = 7.5 with Kd = 30. Watch the front of the bicycle closely, then close the window.
2. The **graph cell** rides Kd = 11 and Kd = 30, both with Kp = 7.5, and draws four graphs: the lean on the top row and the steering command on the bottom row, with Kd = 11 on the left and Kd = 30 on the right. Compare the two bottom graphs carefully, and look at the steering shake in the two reports.

If a window cell stops with a message saying MuJoCo was already imported, use **Kernel → Restart Kernel**. Run the window cell again, then run the Step 1 and Step 2 cells once more before you run the graph cell.

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
cfg = replace(
    cfg,
    pid_kp_roll=7.5,
    pid_ki_roll=0.0,
    pid_kd_roll=30.0,
    pid_kp_steer=1.1164,
    duration=8.0,
)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print(
    f"Kp={cfg.pid_kp_roll:.4f}  Ki={cfg.pid_ki_roll:.4f}  "
    f"Kd={cfg.pid_kd_roll:.4f}  Kp_steer={cfg.pid_kp_steer:.4f}"
)
print("Kp fixed at 7.5, Kd = 30: watch the handlebars")
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")

In [ ]:
kp_now = 7.5
calm_history, calm_report = ride(with_gains(kp=kp_now, ki=0.0, kd=11.0))
shaky_history, shaky_report = ride(with_gains(kp=kp_now, ki=0.0, kd=30.0))

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, constrained_layout=True)
plot_roll(calm_history, f"Kp = {kp_now}, Kd = 11", ax=axes[0, 0])
plot_roll(shaky_history, f"Kp = {kp_now}, Kd = 30", ax=axes[0, 1])
plot_steer(calm_history, f"Kp = {kp_now}, Kd = 11", ax=axes[1, 0])
plot_steer(shaky_history, f"Kp = {kp_now}, Kd = 30", ax=axes[1, 1])
plt.show()

### What Step 6 shows

With Kd = 30, the bicycle still holds and its lean stays small. But the steering command is no longer calm. It flicks back and forth very quickly, often all the way to its 12 rad/s limit in each direction. The steering shake in the report is hundreds of times larger than with Kd = 11.

Here is why. Recall from Lesson 2 that the balancing loop wakes up every 0.02 seconds. With a huge Kd, even a tiny, quick change in the lean produces a big steering command. That steering causes a quick change in the other direction by the next 0.02-second moment, which produces a big command the other way, and so on. The loop is over-reacting to its own corrections.

On a real bicycle this would be a serious problem. The steering motor would buzz back and forth, heat up and wear out, and the real lean sensor is a little noisy, so a huge Kd would turn that small noise into large steering jerks.

This is your first **trade-off**. More D brakes the fall better, but too much D makes the steering shake. A good Kd is large enough to hold the bicycle firmly and still well below the point where shaking begins. On this bicycle, shaking starts to appear somewhere in the mid-20s.

## Step 7. Look closely at a ride that holds

With Kp = 7.5 and a sensible Kd such as 11, the bicycle holds, and its lean graph looks almost flat after the start. But "almost flat" is not the same as "exactly at 0°". To see the difference, we need to zoom in.

In this step you ride Kp = 7.5, Ki = 0, Kd = 11 again, and look at the **zoomed** lean graph. It leaves out the first 2 seconds (the speed-up) and stretches the up-and-down scale to show only −1.5° to +1.5°.

There are two cells:

1. The **window cell** plays the ride. You will probably not see anything wrong in the window, because the effect is a fraction of a degree. That is exactly why engineers use graphs and measurements.
2. The **graph cell** draws the zoomed graph for this ride. It also rides two larger values of Kp, 11 and 14, with the same Kd, prints their reports, and adds them to the same zoomed graph, so that you can compare all three lines.

When you read the zoomed graph, ask yourself: once the line has settled, does it sit on the 0° line, or a little above or below it? Compare that with the leftover lean in the reports. Then compare the three Kp values. Does a larger Kp change the leftover lean? Does it remove it completely?

If a window cell stops with a message saying MuJoCo was already imported, use **Kernel → Restart Kernel**. Run the window cell again, then run the Step 1 and Step 2 cells once more before you run the graph cell.

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
cfg = replace(
    cfg,
    pid_kp_roll=7.5,
    pid_ki_roll=0.0,
    pid_kd_roll=11.0,
    pid_kp_steer=1.1164,
    duration=8.0,
)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print(
    f"Kp={cfg.pid_kp_roll:.4f}  Ki={cfg.pid_ki_roll:.4f}  "
    f"Kd={cfg.pid_kd_roll:.4f}  Kp_steer={cfg.pid_kp_steer:.4f}"
)
print("Kp = 7.5, Ki = 0, Kd = 11: the lean looks steady by eye")
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
for kp in (7.5, 11.0, 14.0):
    history, report = ride(with_gains(kp=kp, ki=0.0, kd=11.0))
    plot_roll(history, "Ki = 0, Kd = 11, zoomed after the start", ax=ax, zoom=True, label=f"Kp = {kp}")
plt.show()

### What Step 7 shows

After the start, the line does not settle on 0°. It settles a few tenths of a degree to one side and stays there. That is the **leftover lean**. A bigger Kp shrinks it, but even at Kp = 14 it does not go away.

Why does it stay? Something small and steady keeps nudging the bicycle toward one side. On a real bicycle it might be a bag hanging on one side, or a frame that is slightly heavier on one side. In this simulation, the bicycle also drifts into a gentle curve as it rides. The exact cause does not matter to the controller. What matters is that the nudge is steady.

Now think about what P does against a steady nudge. P only pushes back when there is a lean, and the size of its push matches the size of the lean. So the bicycle settles at the lean where P's push exactly cancels the nudge. If the lean became 0°, P's push would become 0 too, and the nudge would win again. So P *needs* a little leftover lean in order to push at all. A bigger Kp can make the same push with a smaller lean, which is why the leftover lean shrank, but it can never make it zero. D cannot help either, because once the bicycle has settled, the lean is not changing, so D has nothing to react to.

This is the "heavy bag" picture from Lesson 2, now measured on the real ride. The part of PID that was designed for it is I.

## Step 8. Add I, the memory that removes the leftover lean

Recall from Lesson 2 how I works. It keeps a running total of the lean: every 0.02 seconds, it adds the current lean to the total. While a leftover lean persists, the total keeps growing, so I's piece of the steering command keeps growing too. It keeps growing until the extra steering has pushed the leftover lean away. That is something P cannot do, because P's push shrinks as the lean shrinks, while I's running total holds on to everything it has collected.

In this experiment, **Kp stays at 7.5** and **Kd stays at 11**. Only Ki changes. You will try Ki = 0, 0.25, 0.5, 0.75, 1, 1.5 and 3.

There are two cells:

1. The **window cell** plays Kp = 7.5, Kd = 11 with Ki = 0.75. As in Step 7, the difference is too small to see by eye, so this is mainly to confirm that the ride still looks normal.
2. The **graph cell** rides every Ki value, prints a report for each one, and draws the zoomed lean graph for Ki = 0, 0.75 and 3 on the same axes.

Read the leftover lean in the reports from top to bottom. As Ki grows, does the leftover lean move toward 0? Does it stop at 0, or keep going past 0 to the other side? Then look at the zoomed graph and find the same pattern in the three lines.

If a window cell stops with a message saying MuJoCo was already imported, use **Kernel → Restart Kernel**. Run the window cell again, then run the Step 1 and Step 2 cells once more before you run the graph cell.

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
cfg = replace(
    cfg,
    pid_kp_roll=7.5,
    pid_ki_roll=0.75,
    pid_kd_roll=11.0,
    pid_kp_steer=1.1164,
    duration=8.0,
)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print(
    f"Kp={cfg.pid_kp_roll:.4f}  Ki={cfg.pid_ki_roll:.4f}  "
    f"Kd={cfg.pid_kd_roll:.4f}  Kp_steer={cfg.pid_kp_steer:.4f}"
)
print("Kp = 7.5, Kd = 11, trying Ki = 0.75")
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")

In [ ]:
kp_now, kd_now = 7.5, 11.0
print(f"Ki sweep with Kp fixed at {kp_now:.4f} and Kd fixed at {kd_now:.4f}")
ki_histories = {}
for ki in (0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 3.0):
    ki_histories[ki], report = ride(with_gains(kp=kp_now, ki=ki, kd=kd_now))

fig, ax = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
for ki in (0.0, 0.75, 3.0):
    plot_roll(
        ki_histories[ki],
        f"Kp = {kp_now}, Kd = {kd_now}, zoomed after the start",
        ax=ax,
        zoom=True,
        label=f"Ki = {ki}",
    )
plt.show()

### What Step 8 shows

With Ki = 0, the leftover lean is a few tenths of a degree, as in Step 7. As Ki grows to about 0.5, the leftover lean shrinks to almost exactly 0. That is I doing its job.

But keep going, and the leftover lean passes 0 and grows again on the **other** side. With Ki = 3 it ends up about as large as with no I at all, just on the opposite side. This is **overshoot**.

Here is why too much I overshoots. I's running total does not empty out the instant the lean reaches 0. It still holds everything it collected while the bicycle was leaning, so it keeps steering for a while after the lean is gone. With a large Ki, that leftover memory is strong enough to push the bicycle past upright and out the other side.

So I has its own trade-off. Too little Ki leaves a leftover lean. Too much Ki overshoots and wanders from side to side. A good Ki is the value that brings the leftover lean close to 0 without carrying it past.

## Step 9. Tune all the gains yourself

You now know what every gain does on this bicycle:

- **Kd** catches the fall. Too little and the bicycle falls. Too much and the steering shakes.
- **Kp** sets how firmly the bicycle is pulled back toward upright, and works as a team with Kd. A larger Kp also shrinks the leftover lean, but cannot remove it on its own.
- **Ki** removes the leftover lean. Too much and it overshoots to the other side.
- **Kp_steer** keeps the handlebars from drifting away from straight. Leave it at 1.1164 in this lesson.

Now put it all together and find your own set of gains. Start from the weak gains in `tutorial.yaml` (Kp = 5, Ki = 0, Kd = 6) and improve them using the method from this lesson. Here is a recipe that works well:

1. **Make it hold.** Raise Kd, and if needed Kp, until the bicycle holds.
2. **Make it firm.** Adjust Kd and Kp until the largest lean is small, keeping the steering shake tiny.
3. **Remove the leftover lean.** Raise Ki in small steps until the leftover lean is close to 0. If it passes 0 to the other side, lower Ki again.
4. **Check everything once more.** A change in step 3 can slightly affect steps 1 and 2.

Your targets are the "good values" from Step 2:

| Measurement | Target |
| --- | --- |
| result | held |
| largest lean | under 10° |
| leftover lean | between −0.10° and +0.10° |
| steering shake | under 0.10 rad/s |

The next two cells work as a pair, like the "Try it" cells in Step 5. Edit `kp_try`, `ki_try` and `kd_try` to the **same** values in both cells. Run the window cell to watch the ride, close the window, then run the graph cell. It prints the report, checks each target, and draws both the full and the zoomed lean graphs.

Keep a record of your trials. Engineers always do this, because it is easy to forget which combination gave which result. Double-click the table below to edit it, and add a row for each trial.

| Trial | Kp | Ki | Kd | Result | Largest lean | Leftover lean | Steering shake |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 5 | 0 | 6 | | | | |
| 2 | | | | | | | |
| 3 | | | | | | | |

If a window cell stops with a message saying MuJoCo was already imported, use **Kernel → Restart Kernel**. Run the window cell again, then run the Step 1 and Step 2 cells once more before you run the graph cell.

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


# Keep these three numbers the same as in the graph cell below.
kp_try = 5.0
ki_try = 0.0
kd_try = 6.0
tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = load_config(tutorial_yaml)
cfg = replace(
    cfg,
    pid_kp_roll=float(kp_try),
    pid_ki_roll=float(ki_try),
    pid_kd_roll=float(kd_try),
    pid_kp_steer=1.1164,
    duration=8.0,
)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print(
    f"Kp={cfg.pid_kp_roll:.4f}  Ki={cfg.pid_ki_roll:.4f}  "
    f"Kd={cfg.pid_kd_roll:.4f}  Kp_steer={cfg.pid_kp_steer:.4f}"
)
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")

In [ ]:
# Keep these three numbers the same as in the window cell above.
kp_try = 5.0
ki_try = 0.0
kd_try = 6.0
history, report = ride(with_gains(kp=kp_try, ki=ki_try, kd=kd_try))

if report["fall_time"] is None:
    checks = [
        ("largest lean under 10 deg", report["largest_lean"] < 10.0),
        ("leftover lean between -0.10 and +0.10 deg", abs(report["leftover_lean"]) <= 0.10),
        ("steering shake under 0.10 rad/s", report["steering_shake"] < 0.10),
    ]
    for name, ok in checks:
        print(f"    {'met    ' if ok else 'not yet'}  {name}")
else:
    print("    not yet  the bicycle has to hold before the other targets can be checked")

title = f"Kp = {kp_try}, Ki = {ki_try}, Kd = {kd_try}"
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
plot_roll(history, title, ax=axes[0])
plot_roll(history, title + ", zoomed", ax=axes[1], zoom=True)
plt.show()

## Step 10. Save your gains

So far, your gains have only been used inside this notebook. Anything that exists only in the notebook's memory disappears when the kernel is restarted or the notebook is closed. To keep your gains, write them into the settings file on the disk.

1. Open `configs/tutorial.yaml` in the editor. It is in the `configs` folder, next to the `tutorials` folder.
2. Find the four lines that start with `pid_kp_roll`, `pid_ki_roll`, `pid_kd_roll` and `pid_kp_steer`.
3. Change the first three to your best Kp, Ki and Kd from Step 9. Keep `pid_kp_steer` at 1.1164. Four decimal places are plenty. For example, if your best trial was Kp = 8, Ki = 0.6 and Kd = 12, the lines would read:

```yaml
pid_kp_roll: 8.0
pid_ki_roll: 0.6
pid_kd_roll: 12.0
pid_kp_steer: 1.1164
```

4. Save the file with Ctrl+S (Cmd+S on macOS).

Then run the next cell. It reads `tutorial.yaml` fresh from the disk, so it shows what is really saved in the file rather than what is in the notebook's memory, and it rides those gains once more. The first line of the report should show your gains. If it still shows Kp = 5, Ki = 0 and Kd = 6, the file has not been saved with the new values yet. Edit it, save it, and run the cell again.

In [ ]:
saved_cfg = load_config(tutorial_yaml)
print("Read from", os.path.relpath(tutorial_yaml, project_root))
saved_history, saved_report = ride(saved_cfg)

## Step 11. Compare your gains with a reference set

Now that you have tuned your own gains, here is a set to compare them with. It was found earlier with a much longer search on this same bicycle: the same idea as your trial and error, but with hundreds of trials. It is stored in `configs/new_bike.yaml`, the file that Lesson 1 used.

There are three cells:

1. The **first window cell** plays your saved gains from `tutorial.yaml`. Watch it, then close the window.
2. The **second window cell** plays the reference set from `new_bike.yaml`. Its first printed line shows the reference gains. Watch it, then close the window.
3. The **graph cell** rides both sets, prints a report for each, and draws them side by side, with the full lean graph on the top row and the zoomed graph on the bottom row.

Compare the two reports measurement by measurement. Where is yours better, where is the reference better, and where are they about the same?

If a window cell stops with a message saying MuJoCo was already imported, use **Kernel → Restart Kernel**. Run the window cell again, then run the Step 1 and Step 2 cells once more before you run the graph cell.

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
cfg = replace(load_config(tutorial_yaml), duration=8.0)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print("Your saved gains from", os.path.relpath(tutorial_yaml, project_root))
print(
    f"Kp={cfg.pid_kp_roll:.4f}  Ki={cfg.pid_ki_roll:.4f}  "
    f"Kd={cfg.pid_kd_roll:.4f}  Kp_steer={cfg.pid_kp_steer:.4f}"
)
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = replace(load_config(new_bike_yaml), duration=8.0)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print("Reference gains from", os.path.relpath(new_bike_yaml, project_root))
print(
    f"Kp={cfg.pid_kp_roll:.4f}  Ki={cfg.pid_ki_roll:.4f}  "
    f"Kd={cfg.pid_kd_roll:.4f}  Kp_steer={cfg.pid_kp_steer:.4f}"
)
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")

In [ ]:
print("Your gains (tutorial.yaml):")
yours_history, yours_report = ride(load_config(tutorial_yaml))
print("Reference gains (new_bike.yaml):")
reference_history, reference_report = ride(load_config(new_bike_yaml))

fig, axes = plt.subplots(2, 2, figsize=(11, 6), constrained_layout=True)
plot_roll(yours_history, "your gains", ax=axes[0, 0])
plot_roll(reference_history, "reference gains", ax=axes[0, 1])
plot_roll(yours_history, "your gains, zoomed", ax=axes[1, 0], zoom=True)
plot_roll(reference_history, "reference gains, zoomed", ax=axes[1, 1], zoom=True)
plt.show()

### Reading the comparison

The reference set is:

| Gain | Reference value |
| --- | --- |
| Kp (`pid_kp_roll`) | 7.5356 |
| Ki (`pid_ki_roll`) | 1.2615 |
| Kd (`pid_kd_roll`) | 11.2605 |
| Kp_steer (`pid_kp_steer`) | 1.1164 |

Compare these values with what you discovered yourself:

- **Kd** is well above the point where the bicycle starts to hold (about 6) and well below the point where the steering starts to shake (the mid-20s). That is a comfortable middle.
- **Kp** is in the same region you explored.
- **Ki** is a little larger than the values that removed the leftover lean best in Step 8. Look at its leftover lean in the report: it has overshot slightly to the other side. By this lesson's leftover-lean target, the reference set does not quite pass.

That last point surprises many students, so it is worth explaining. The longer search scored rides in its own way, weighing several things at once, and the leftover lean over the last 3 seconds was not the only thing it cared about. So it settled on slightly different values from the ones this lesson's targets would pick. Your gains may well beat the reference on one or two measurements. That is normal, and it does not mean either set is wrong. In engineering there is rarely one perfect answer, only good answers for a chosen set of goals.

**Which set should you keep for Lessons 4 and 5?** If your gains met all four targets in Step 9, keep them in `tutorial.yaml`: they are your own work. If they did not, or if the bicycle fell, copy the reference set into `tutorial.yaml` instead. It holds the bicycle firmly, which is what the next lessons need. Save the file, and run the Step 10 cell again to confirm:

```yaml
pid_kp_roll: 7.5356
pid_ki_roll: 1.2615
pid_kd_roll: 11.2605
pid_kp_steer: 1.1164
```

## Questions

These questions check whether you can predict what a gain change will do before you see the result. Each one describes a ride. Write down your prediction for each, then run the next cell, which rides all four and prints their reports in order.

1. A bicycle with Kp = 7.5, Ki = 0, Kd = 11 holds with a leftover lean of about +0.3°. A classmate changes only Ki, to 0.5. Will the leftover lean move closer to 0, stay the same, or grow?
2. Another classmate wants a really steady bicycle and sets Kd = 40 (with Kp = 7.5, Ki = 0). Will the bicycle fall? If it holds, what do you expect the steering shake to be: tiny, or large?
3. With Kd = 11 and Ki = 0, Kp is lowered from 7.5 to 3. Will the bicycle still hold? If it holds, will the leftover lean be larger or smaller than at Kp = 7.5?
4. With Kp = 7.5 and Kd = 11, Ki is raised all the way to 3. Which side of 0 do you expect the leftover lean to end up on: the same side as with Ki = 0, or the other side?

In [ ]:
questions = [
    ("Question 1", 7.5, 0.5, 11.0),
    ("Question 2", 7.5, 0.0, 40.0),
    ("Question 3", 3.0, 0.0, 11.0),
    ("Question 4", 7.5, 3.0, 11.0),
]
for name, kp, ki, kd in questions:
    print(name)
    ride(with_gains(kp=kp, ki=ki, kd=kd))

## Discussion

Here are some questions students often ask after this lesson, with explanations.

### Why can't P balance the bicycle on its own, however large Kp is?

Because P only reacts to how far the bicycle has already leaned, never to how fast it is falling. At the start of a fall the lean is small, so P's push is small, while the bicycle is already picking up speed. By the time the lean is large enough for P to push hard, it is too late. A larger Kp pushes harder at every angle, but it is still always late. D fixes this because it reacts to the falling speed, which is already large at the very start.

### If more D gave a smaller lean, why not use the largest D possible?

Because of the steering shake you saw in Step 6. With a huge Kd, the loop reacts so strongly to tiny, quick changes in the lean that each correction causes a new quick change the other way, and the handlebars end up flicking back and forth at their limit. A real motor would buzz, heat up and wear out, and the small noise in a real lean sensor would be turned into large steering jerks. A good Kd brakes the fall firmly while staying well away from that region.

### Why does the bicycle keep a small leftover lean without I?

Something small and steady keeps nudging the bicycle to one side. P can only push back when there is a lean, and its push grows with the lean, so the bicycle settles where P's push exactly cancels the nudge. That point is never exactly 0°, because at 0° P would not push at all. A larger Kp shrinks the leftover lean but cannot remove it. D cannot help either, because a settled lean is not changing.

### Why does I remove the leftover lean, and why does too much I make it worse?

I keeps a running total of the lean, so a lean that lasts keeps adding to I's push until the lean is gone. That is something P cannot do. But the total does not empty out the moment the lean reaches 0, so I keeps pushing for a while afterwards. A small Ki pushes gently enough to stop close to 0. A large Ki carries the bicycle past 0 to the other side, which is the overshoot you saw with Ki = 3.

### Why does the right value of one gain depend on the others?

Because all the parts add their pushes together into one steering command. In Lesson 2, Kd = 6 fell with Kp = 5. In this lesson, Kd = 6 held with Kp = 7.5. The stronger P was doing part of the work, so less D was needed. This is why each report prints all four gains: a single gain value only makes sense together with the others it was tested with.

### My gains are different from the reference set. Did I do something wrong?

No. Tuning almost never has a single correct answer. Different sets of gains can all keep the bicycle up well, each a little better at one thing and a little worse at another. The reference set was found by a longer search that scored rides in its own way. If your gains met the targets in Step 9, you have tuned the balancing loop successfully, using the same method engineers use on real machines.